# RAG Part 1

This notebook walks you through how we study and validate the impact on performance between
1. [Section-aware chunking vs Fixed-size chunking](#section-1-section-aware-chunking-vs-fixed-size-chunking)
2. [Dense Retrieval vs Sparse Retrieval](#section-2-dense-retrieval-vs-sparse-retrieval)
3. [With vs Without Re-ranking](#section-3-with-vs-without-re-ranking)

## Section 0: Setup

In [1]:
import sys
print(sys.executable)

c:\Users\Min Yee\Documents\GitHub\smu-llm-group-project\.venv\Scripts\python.exe


In [2]:
import sys
from pathlib import Path
# Add project root to sys.path so 'src' can be imported
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [3]:
#%pip install torch
import torch
torch.cuda.is_available()

True

In [10]:
import pickle
import os

# Paths for BM25
bm25_fixed_path = os.path.join('..', 'data', 'bm25_index', 'bm25_index', 'fixed_500char.pkl')
bm25_section_path = os.path.join('..', 'data', 'bm25_index', 'bm25_index', 'section_400token.pkl')

# Paths for tokenized corpus (new)
bm25_fixed_tokens_path = os.path.join('..', 'data', 'bm25_index', 'bm25_index', 'fixed_500char_tokens.pkl')
bm25_section_tokens_path = os.path.join('..', 'data', 'bm25_index', 'bm25_index', 'section_400token_tokens.pkl')

# Load BM25 chunks
with open(bm25_fixed_path, 'rb') as f:
    bm25_fixed = pickle.load(f)
with open(bm25_section_path, 'rb') as f:
    bm25_section = pickle.load(f)

# Load tokenized corpus (if exists)
if os.path.exists(bm25_fixed_tokens_path):
    with open(bm25_fixed_tokens_path, 'rb') as f:
        bm25_fixed_tokens = pickle.load(f)
else:
    bm25_fixed_tokens = None

if os.path.exists(bm25_section_tokens_path):
    with open(bm25_section_tokens_path, 'rb') as f:
        bm25_section_tokens = pickle.load(f)
else:
    bm25_section_tokens = None

# Paths for Vector Store (Chroma)
vector_fixed_path = os.path.join('..', 'data', 'vector_store', 'vector_store', 'fixed_500char', 'pubmedbert', 'chroma.sqlite3')
vector_section_path = os.path.join('..', 'data', 'vector_store', 'vector_store', 'section_400token', 'pubmedbert', 'chroma.sqlite3')

# For vector stores, you may need to use ChromaDB or similar library to load the DB files
# Example placeholder (replace with actual loading code as needed):
vector_fixed = vector_fixed_path
vector_section = vector_section_path

In [11]:
# --- Save tokenized corpus if missing ---
import pickle
import os

def save_tokenized_corpus(bm25_obj, tokens_path, fallback_texts=None):
    if os.path.exists(tokens_path):
        print(f"Tokenized corpus already exists: {tokens_path}")
        return
    # Try to extract from BM25 object (if possible)
    if hasattr(bm25_obj, 'corpus') and bm25_obj.corpus:
        tokenized = bm25_obj.corpus
    elif fallback_texts is not None:
        # Tokenize fallback_texts (list of strings)
        tokenized = [t.split() for t in fallback_texts]
    else:
        raise ValueError("No tokenized corpus found and no fallback texts provided.")
    with open(tokens_path, 'wb') as f:
        pickle.dump(tokenized, f)
    print(f"Saved tokenized corpus to: {tokens_path}")

# Example usage:
# If you have the original chunk texts, provide them as fallback_texts
# fallback_texts = [...]  # List of original chunk strings
# save_tokenized_corpus(bm25_fixed, bm25_fixed_tokens_path, fallback_texts)
# save_tokenized_corpus(bm25_section, bm25_section_tokens_path, fallback_texts)


## Section 1: Section-aware chunking vs Fixed-size chunking

In [12]:
# --- RAG and Evaluation for All 4 Chunk Types ---
import time
from rank_bm25 import BM25Okapi
from pathlib import Path
from src.chroma_manager import ChromaManager
from sentence_transformers import SentenceTransformer

# Dummy query and ground truth for demonstration
queries = ["What is the main finding of the study?"]
ground_truths = ["The main finding is ..."]

# Helper: Evaluate retrieval (simple accuracy for demo)
def evaluate_retrieval(results, ground_truths):
    # Placeholder: count if any retrieved doc contains ground truth string
    correct = 0
    for res, gt in zip(results, ground_truths):
        if any(gt in doc for doc in res):
            correct += 1
    return correct / len(ground_truths)

# --- BM25 Retrieval ---
def bm25_retrieve(bm25, tokenized_corpus, queries, topk=3):
    results = []
    for q in queries:
        tokenized_q = q.split()
        scores = bm25.get_scores(tokenized_q)
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:topk]
        docs = [" ".join(tokenized_corpus[i]) for i in top_idx]
        results.append(docs)
    return results

# --- Vector Retrieval ---
def vector_retrieve(chroma_path, queries, topk=3):
    # Load ChromaDB collection
    chroma = ChromaManager(
        base_directory=Path(chroma_path).parent.parent,
        chunk_strategy=Path(chroma_path).parent.name,
        embedding_model="pubmedbert"
    )
    results = []
    for q in queries:
        # This assumes a method like chroma.query exists (adjust as needed)
        docs = chroma.query(q, top_k=topk)
        results.append([d["text"] for d in docs])
    return results

# --- Run and Evaluate ---
start = time.time()
bm25_fixed_results = bm25_retrieve(bm25_fixed, bm25_fixed_tokens, queries)
bm25_fixed_score = evaluate_retrieval(bm25_fixed_results, ground_truths)
bm25_section_results = bm25_retrieve(bm25_section, bm25_section_tokens, queries)
bm25_section_score = evaluate_retrieval(bm25_section_results, ground_truths)

vector_fixed_results = vector_retrieve(vector_fixed, queries)
vector_fixed_score = evaluate_retrieval(vector_fixed_results, ground_truths)
vector_section_results = vector_retrieve(vector_section, queries)
vector_section_score = evaluate_retrieval(vector_section_results, ground_truths)
end = time.time()

print(f"BM25 Fixed Score: {bm25_fixed_score}")
print(f"BM25 Section Score: {bm25_section_score}")
print(f"Vector Fixed Score: {vector_fixed_score}")
print(f"Vector Section Score: {vector_section_score}")
print(f"Total evaluation time: {end-start:.2f} seconds")

TypeError: 'NoneType' object is not subscriptable

## Section 2: Dense Retrieval vs Sparse Retrieval

## Section 3: With vs Without Re-ranking